# Exercise 4: Implement Factory Pattern for Data Transformations

### Objective:
- Practice the **Factory Pattern** for scalable creation of transformation objects.
- Apply **abstraction** and **composition** in Data Science workflows.

### Concept:
Instead of tightly coupling user scripts to specific concrete transformation classes (or maintaining complex instantiation logic everywhere), a **Factory** provides a centralized method that instantiates and returns the appropriate transformation object based on a simple name or configuration string.

## 1. Defining the Abstract DataTransform Interface

We create an abstract base class `DataTransform` with the abstract method `apply(df)`.

In [1]:
import pandas as pd
from abc import ABC, abstractmethod

class DataTransform(ABC):
    """
    Abstract Base Class for all data transformation operations.
    """
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies transformation logic to the input DataFrame.
        Must return a new or modified DataFrame.
        """
        pass

## 2. Implementing Concrete Transformation Subclasses

We implement three transformations:
1. `NormalizeColumns`: Scales numerical columns to the range [0, 1] using Min-Max scaling.
2. `RemoveDuplicates`: Drops duplicate rows from the dataset.
3. `StandardizeText`: Converts text/string columns to lower-case and strips leading/trailing whitespaces.

In [2]:
class NormalizeColumns(DataTransform):
    """Normalizes numerical columns using Min-Max normalization [0, 1]."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[Transform] Applying NormalizeColumns (Min-Max Scaling)...")
        transformed_df = df.copy()
        
        # Select numeric columns (excluding 'id' if present, to keep identifier intact)
        num_cols = [c for c in transformed_df.select_dtypes(include=['number']).columns if c != 'id']
        
        for col in num_cols:
            col_min = transformed_df[col].min()
            col_max = transformed_df[col].max()
            if col_max > col_min:
                transformed_df[col] = (transformed_df[col] - col_min) / (col_max - col_min)
                print(f"  - Scaled '{col}' from [{col_min:.1f}, {col_max:.1f}] to [0, 1]")
                
        return transformed_df


class RemoveDuplicates(DataTransform):
    """Removes duplicate rows from the DataFrame."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[Transform] Applying RemoveDuplicates...")
        before_count = len(df)
        transformed_df = df.drop_duplicates(subset=[c for c in df.columns if c != 'id']).copy()
        after_count = len(transformed_df)
        print(f"  - Removed {before_count - after_count} duplicate rows (from {before_count} to {after_count})")
        return transformed_df


class StandardizeText(DataTransform):
    """Standardizes text columns to lowercase and strips whitespace."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[Transform] Applying StandardizeText...")
        transformed_df = df.copy()
        text_cols = transformed_df.select_dtypes(include=['object', 'string']).columns
        
        for col in text_cols:
            transformed_df[col] = transformed_df[col].astype(str).str.strip().str.lower()
            print(f"  - Standardized text in column '{col}'")
            
        return transformed_df

## 3. Creating the TransformFactory

The `TransformFactory` provides a centralized `get_transform()` method.
This decouples client code from the concrete classes.

In [3]:
class TransformFactory:
    """
    Factory class that returns transformation objects based on string identifiers.
    """
    @staticmethod
    def get_transform(transform_name: str) -> DataTransform:
        key = transform_name.lower().strip()
        
        if key in ["normalize", "normalize_columns"]:
            return NormalizeColumns()
        elif key in ["remove_duplicates", "deduplicate"]:
            return RemoveDuplicates()
        elif key in ["standardize_text", "clean_text"]:
            return StandardizeText()
        else:
            raise ValueError(f"Unknown transformation: '{transform_name}'. "
                             f"Available: ['normalize', 'remove_duplicates', 'standardize_text']")

## 4. Testing the Factory Pattern with Data Transformations

In [4]:
# Load sample data (and fill NAs for demonstration)
df_sample = pd.read_csv("sample_data.csv")
# Quick imputation so transformations work smoothly
df_sample['age'] = df_sample['age'].fillna(df_sample['age'].mean())
df_sample['salary'] = df_sample['salary'].fillna(df_sample['salary'].mean())
df_sample['city'] = df_sample['city'].fillna('Unknown')

print("Starting Data:")
display(df_sample)

Starting Data:


,id,name,age,salary,city
0,1,Alice,25.0,50000.000000,New York
1,2,Bob,30.0,62142.857143,Paris
2,3,Charlie,30.5,70000.000000,London
3,4,David,40.0,90000.000000,TOKYO
4,5,Alice,25.0,50000.000000,New York
5,6,Eve,35.0,65000.000000,paris
6,7,Frank,28.0,52000.000000,london
7,8,Grace,30.5,58000.000000,Unknown


In [5]:
# 1. Use factory to get 'remove_duplicates'
dedup_transform = TransformFactory.get_transform("remove_duplicates")
df_dedup = dedup_transform.apply(df_sample)
display(df_dedup)

[Transform] Applying RemoveDuplicates...
  - Removed 1 duplicate rows (from 8 to 7)


,id,name,age,salary,city
0,1,Alice,25.0,50000.000000,New York
1,2,Bob,30.0,62142.857143,Paris
2,3,Charlie,30.5,70000.000000,London
3,4,David,40.0,90000.000000,TOKYO
5,6,Eve,35.0,65000.000000,paris
6,7,Frank,28.0,52000.000000,london
7,8,Grace,30.5,58000.000000,Unknown


In [6]:
# 2. Use factory to get 'standardize_text'
text_transform = TransformFactory.get_transform("standardize_text")
df_text = text_transform.apply(df_dedup)
display(df_text)

[Transform] Applying StandardizeText...
  - Standardized text in column 'name'
  - Standardized text in column 'city'


,id,name,age,salary,city
0,1,alice,25.0,50000.000000,new york
1,2,bob,30.0,62142.857143,paris
2,3,charlie,30.5,70000.000000,london
3,4,david,40.0,90000.000000,tokyo
5,6,eve,35.0,65000.000000,paris
6,7,frank,28.0,52000.000000,london
7,8,grace,30.5,58000.000000,unknown


In [7]:
# 3. Use factory to get 'normalize'
norm_transform = TransformFactory.get_transform("normalize")
df_norm = norm_transform.apply(df_text)
display(df_norm)

[Transform] Applying NormalizeColumns (Min-Max Scaling)...
  - Scaled 'age' from [25.0, 40.0] to [0, 1]
  - Scaled 'salary' from [50000.0, 90000.0] to [0, 1]


,id,name,age,salary,city
0,1,alice,0.000000,0.000000,new york
1,2,bob,0.333333,0.303571,paris
2,3,charlie,0.366667,0.500000,london
3,4,david,1.000000,1.000000,tokyo
5,6,eve,0.666667,0.375000,paris
6,7,frank,0.200000,0.050000,london
7,8,grace,0.366667,0.200000,unknown


### Summary:
- Implemented `DataTransform` hierarchy with three concrete transformers.
- Created `TransformFactory` to instantiate objects based on simple string inputs.
- Client code can request transformations dynamically without depending on concrete classes.